In [ ]:
# 交叉熵
import numpy as np
def cross_entropy(logits, target):
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    softmax_probs = exps / np.sum(exps, axis=1, keepdims=True)
    n = logits.shape[0]
    correct_logprobs = -np.log(softmax_probs[range(n),target] + 1e-12)
    loss = np.sum(correct_logprobs) / n
    return loss

def kmeans(x, k, max_iters=100, tol=1e-4):
    n_samples, n_features = x.shape
    indices = np.random.choice(n_samples, k, replace=False)
    centroids = x[indices]

    for i in range(max_iters):
        distances = np.linalg.norm(x[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([x[labels==j].mean(axis=0) if len(x[labels==j]) > 0 else centroids[j] for j in range(k)])
        if np.linalg.norm(new_centroids - centroids) < tol:
            print(f"Converged at iteration {i}")
            break
        centroids = new_centroids
    return centroids, labels

def AUC(y_score, y_true):
    indices = sorted(range(len(y_score)), key=lambda i:y_score[i])
    sorted_labels = [y_true[i] for i in indices]
    pos_c = np.sum(y_true)
    neg_c = len(y_true) - pos_c
    rank_sum = 0
    for i in range(len(sorted_labels)):
        if sorted_labels[i] == 1:
            rank_sum += (i+1)
    auc = (rank_sum - pos_c * (pos_c + 1)/2)/(pos_c * neg_c)
    return auc

# 无提示手撕MHA
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
class MHA_with_cache(nn.Module):
    def __init__(self, d_model, n_heads, bias=True, dropout=0.01):
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=bias)
        self.W_K = nn.Linear(d_model, d_model, bias=bias)
        self.W_V = nn.Linear(d_model, d_model, bias=bias)
        self.W_O = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv, use_cache=True, mask=None):
        B,L,D = x.shape
        Q = self.W_Q(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        K = self.W_K(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        V = self.W_V(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)

        if past_kv:
            pastk, pastv = past_kv
            K = torch.cat([pastk, K], dim=2)
            V = torch.cat([pastv, V], dim=2)
        present_kv = (K,V) if use_cache else None

        score = torch.matmul(Q, K.transpose(-1,-2)) * torch.rsqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask==0, float('-inf'))
        # 错误点1: dropout是对softmax后的结果运算
        attn = self.dropout(F.softmax(score, dim=-1))
        # attn = torch.matmul(F.softmax(score, dim=-1), V)
        # attn = self.dropout(attn)
        attn = torch.matmul(attn, V)
        attn = attn.transpose(1,2).contiguous().view(B,L,D)
        return self.W_O(attn), present_kv
